<a href="https://colab.research.google.com/github/adib422/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Unit of Analysis + Time Window

**Unit of Analysis**

One row represents the daily search and analytics performance of one content page (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).

**Time Window**

The analysis uses a mid-panel month for development to avoid leakage from the final month of the dataset.

**Output**

The objective is to prepare reliable features that can later be used to predict declining content performance.

In [8]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Warehouse connected successfully!")

Warehouse connected successfully!


In [9]:
query = f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date,
COUNT(*) AS rows
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Field Categories

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- visible_queries
- top_query_share

These are available before the refresh decision and can help estimate which pages deserve attention.

### Label

A proxy label (is_declining) indicating whether a content page's impressions in the current 30-day window dropped by more than 20% compared with the previous 30-day window. This is an observed outcome used for supervised learning.

### Context

- client_hash_id
- content_hash_id
- report_date

These identify content and provide the time context but are not prediction targets.

### Excluded

I exclude future performance metrics and any variables created directly from the prediction window because they would introduce feature leakage. Personally identifying information is also excluded because the dataset is pseudonymized.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the fields that will be used

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
LIMIT 5
"""

con.sql(query).df()


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,2025-01-27,30,0,3.833333
1,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,2025-01-27,5,0,71.600000
2,client_9958f0a7ae1df715,content_b4462a1b90640058,2025-01-27,1,0,34.000000
3,client_9958f0a7ae1df715,content_c899aef92518c714,2025-01-27,6,0,23.333333
4,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,2025-01-27,5,0,17.800000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries

To validate my data contract, I performed three verification queries on the March 2026 partition.

1. Verify the grain (one row represents one client, one content item, and one report date).
2. Verify the date range and total number of rows for the selected month.
3. Verify that rows with Search Console availability (`is_active IS TRUE`) are available for analysis.

These checks confirm that the data follows the expected structure before feature engineering and model development.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------
# Query 1 : Verify the grain
# ---------------------------------------------------

print("="*60)
print("Query 1 : Grain Verification")
print("="*60)

grain = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

print(grain)

if len(grain)==0:
    print("\n✅ Grain Verified : One row = one client + one content page + one report date")


# ---------------------------------------------------
# Query 2 : Count + Date Window
# ---------------------------------------------------

print("\n")
print("="*60)
print("Query 2 : Row Count + Date Window")
print("="*60)

window = con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date,
COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
""").df()

display(window)


# ---------------------------------------------------
# Query 3 : Availability Check
# ---------------------------------------------------

print("\n")
print("="*60)
print("Query 3 : Active Clients (IS TRUE)")
print("="*60)

availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM {TABLES['fact_daily']} f
JOIN {TABLES['dim_clients']} c
ON f.client_hash_id = c.client_hash_id
WHERE
strftime(report_date,'%Y-%m')='2026-03'
AND c.is_active IS TRUE
""").df()

display(availability)


Query 1 : Grain Verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, duplicate_rows]
Index: []

✅ Grain Verified : One row = one client + one content page + one report date


Query 2 : Row Count + Date Window


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378




Query 3 : Active Clients (IS TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,7864344


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

Although this warehouse provides a rich history of Search Console performance, it has several limitations.

- Client histories are unbalanced because different clients started collecting data at different times.
- Early months may contain only Search Console data while Google Analytics data may not yet be available.
- The warehouse contains pseudonymized identifiers, so individual websites or users cannot be identified.
- Features created from future time windows cannot be used for prediction because they would introduce data leakage.
- External factors such as Google algorithm updates, seasonal events, marketing campaigns, or competitor actions are not represented in the dataset.

These limitations should be considered when interpreting model performance and deploying prediction models.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check client history ranges

history = con.sql(f"""
SELECT
    client_hash_id,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_days
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id
ORDER BY first_date
LIMIT 10
""").df()

history


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,total_days
0,client_9958f0a7ae1df715,2025-01-27,2026-06-30,1470881
1,client_ff644d8251367cbb,2025-01-27,2026-06-30,1246888
2,client_73cda7b4e4f265ea,2025-02-11,2026-06-30,8708971
3,client_fef1a8f436438636,2025-03-11,2026-06-30,2779933
4,client_62f4a7e64f5e0096,2025-06-07,2026-06-30,5676451
5,client_b10cb2997d0c7c86,2025-06-18,2026-06-30,1050146
6,client_65de48885f4ef01b,2025-06-21,2026-06-30,3296200
7,client_c182d11e4862a37d,2025-06-21,2026-06-30,282410
8,client_3197e6291363b4db,2025-06-29,2026-06-30,2598532
9,client_625b6439094e23e4,2025-07-01,2026-06-30,7589247


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.